# Profilage DVF 2022 — Etape 2f : transactions a faible valeur

L'etape 2c a identifie 33 transactions a 0 euro et des cessions a 1 euro.
La mandante demande d'elargir l'investigation : quel pourcentage de transactions
ont une valeur inferieure a 1 000 euros ? Quel est leur profil ? Sont-elles
des erreurs, des cessions symboliques ou des transactions plausibles ?

Ce notebook examine :

1. Nombre et pourcentage des transactions a moins de 1 000 euros
2. Profil par type de bien et nature de mutation
3. Repartition par tranche de prix (0-10, 10-100, 100-500, 500-1000)
4. Apercu des cas concrets

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [2]:
import duckdb
from pathlib import Path

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
nb_avec_prix = con.execute(f"SELECT count(*) FROM '{pq}' WHERE \"Valeur fonciere\" IS NOT NULL AND \"Valeur fonciere\" > 0").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))
print(f"Lignes avec prix > 0 : {nb_avec_prix:,}".replace(",", " "))

Fichier : dvf-2022.parquet
Lignes : 4 617 590
Lignes avec prix > 0 : 4 586 415


## Cellule 3 — Nombre et pourcentage des transactions a moins de 1 000 euros

Combien de lignes ont un prix strictement positif mais inferieur a 1 000 euros ?
Quel pourcentage cela represente-t-il sur l'ensemble des lignes avec prix ?

In [3]:
seuils = [1, 10, 100, 500, 1000, 5000, 10000]

print(f"{'Seuil':<25} {'Nb lignes':>12} {'% du fichier':>14}")
print("-" * 54)

for s in seuils:
    r = con.execute(f"""
        SELECT COUNT(*)
        FROM '{pq}'
        WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < {s}
    """).fetchone()[0]
    pct = 100.0 * r / nb_avec_prix
    print(f"  < {s:>6,} EUR {r:>12,} {pct:>12.2f} %".replace(",", " "))

Seuil                        Nb lignes   % du fichier
------------------------------------------------------
  <      1 EUR            0         0.00 %
  <     10 EUR       31 312         0.68 %
  <    100 EUR       37 212         0.81 %
  <    500 EUR       67 225         1.47 %
  <  1 000 EUR       99 989         2.18 %
  <  5 000 EUR      265 529         5.79 %
  < 10 000 EUR      374 625         8.17 %


## Cellule 4 — Profil par type de bien et nature de mutation

Pour les transactions a moins de 1 000 euros (prix > 0), quel est le profil ?
Terrains nus, dependances, cessions entre collectivites ?

In [4]:
print("Transactions a moins de 1 000 euros (prix > 0) :")
print()

# Par type de local
print("Par type de local :")
r = con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)') AS type_local,
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < 1000
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchall()
for typ, nb, pct in r:
    print(f"  {typ:<45} {nb:>8,} ({pct:>5.1f} %)".replace(",", " "))

print()
print("Par nature de mutation :")
r2 = con.execute(f"""
    SELECT
        \"Nature mutation\",
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < 1000
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchall()
for nat, nb, pct in r2:
    print(f"  {nat:<45} {nb:>8,} ({pct:>5.1f} %)".replace(",", " "))

Transactions a moins de 1 000 euros (prix > 0) :

Par type de local :
  (vide)                                          89 052 ( 89.1 %)
  Dépendance                                       4 818 (  4.8 %)
  Appartement                                      2 780 (  2.8 %)
  Local industriel. commercial ou assimilé         2 089 (  2.1 %)
  Maison                                           1 250 (  1.3 %)

Par nature de mutation :
  Vente                                           89 103 ( 89.1 %)
  Echange                                         10 439 ( 10.4 %)
  Expropriation                                      227 (  0.2 %)
  Vente terrain à bâtir                              106 (  0.1 %)
  Vente en l'état futur d'achèvement                  70 (  0.1 %)
  Adjudication                                        44 (  0.0 %)


## Cellule 5 — Repartition par tranche de prix

Les transactions a moins de 1 000 euros sont-elles concentrees a 1 euro (cessions symboliques)
ou reparties sur toute la plage ?

In [5]:
tranches = [
    ("1 EUR exactement", "\"Valeur fonciere\" = 1"),
    ("2 a 9 EUR",        "\"Valeur fonciere\" BETWEEN 2 AND 9"),
    ("10 a 99 EUR",      "\"Valeur fonciere\" BETWEEN 10 AND 99"),
    ("100 a 499 EUR",    "\"Valeur fonciere\" BETWEEN 100 AND 499"),
    ("500 a 999 EUR",    "\"Valeur fonciere\" BETWEEN 500 AND 999"),
]

total_lt1000 = con.execute(f"SELECT COUNT(*) FROM '{pq}' WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < 1000").fetchone()[0]

print(f"{'Tranche':<25} {'Nb lignes':>12} {'% des < 1000':>14}")
print("-" * 54)
for label, cond in tranches:
    r = con.execute(f"SELECT COUNT(*) FROM '{pq}' WHERE {cond}").fetchone()[0]
    pct = 100.0 * r / total_lt1000 if total_lt1000 > 0 else 0
    print(f"  {label:<23} {r:>10,} {pct:>12.1f} %".replace(",", " "))
print(f"  {'Total < 1 000 EUR':<23} {total_lt1000:>10,}".replace(",", " "))

Tranche                      Nb lignes   % des < 1000
------------------------------------------------------
  1 EUR exactement            30 459         30.5 %
  2 a 9 EUR                      853          0.9 %
  10 a 99 EUR                  5 900          5.9 %
  100 a 499 EUR               30 013         30.0 %
  500 a 999 EUR               32 764         32.8 %
  Total < 1 000 EUR           99 989


## Cellule 6 — Apercu des transactions a 1 euro

Les cessions a 1 euro sont les plus suspectes. Quels types de biens ? Quelles surfaces ?
Quelles communes ?

In [6]:
r = con.execute(f"""
    SELECT
        \"Date mutation\",
        \"Commune\",
        \"Code departement\",
        COALESCE(\"Type local\", '(vide)') AS type_local,
        \"Surface reelle bati\",
        \"Surface terrain\",
        \"Nature mutation\"
    FROM '{pq}'
    WHERE \"Valeur fonciere\" = 1
    ORDER BY \"Date mutation\"
    LIMIT 20
""").fetchall()

print(f"  {'Date':<12} {'Commune':<20} {'Dept':>4} {'Type local':<25} {'Surf.batie':>10} {'Surf.terr':>10} {'Nature':>10}")
print(f"  {'-'*95}")
for date, com, dept, typ, sb, st, nat in r:
    sb_s = f"{sb:,}".replace(",", " ") if sb is not None else '(null)'
    st_s = f"{st:,}".replace(",", " ") if st is not None else '(null)'
    print(f"  {str(date):<12} {com:<20} {dept:>4} {typ:<25} {sb_s:>10} {st_s:>10} {nat:>10}")

  Date         Commune              Dept Type local                Surf.batie  Surf.terr     Nature
  -----------------------------------------------------------------------------------------------
  2022-01-03   PLOMEUR                29 (vide)                        (null)         10      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  2022-01-03   RIMOGNE                08 Dépendance                         0      7 580      Vente
  

## Cellule 7 — Apercu des transactions entre 100 et 999 euros

Les transactions entre 100 et 999 euros sont-elles des terrains nus de petite taille
(plausible) ou des biens batis (suspect) ?

In [7]:
r = con.execute(f"""
    SELECT
        \"Date mutation\",
        \"Commune\",
        \"Code departement\",
        COALESCE(\"Type local\", '(vide)') AS type_local,
        \"Surface reelle bati\",
        \"Surface terrain\",
        \"Valeur fonciere\",
        \"Nature mutation\"
    FROM '{pq}'
    WHERE \"Valeur fonciere\" BETWEEN 100 AND 999
    ORDER BY \"Valeur fonciere\" ASC
    LIMIT 20
""").fetchall()

print(f"  {'Date':<12} {'Commune':<18} {'Dept':>4} {'Type':<20} {'Surf.bat':>8} {'Surf.ter':>8} {'Prix':>8} {'Nature':>8}")
print(f"  {'-'*90}")
for date, com, dept, typ, sb, st, vf, nat in r:
    sb_s = f"{sb}" if sb is not None else '-'
    st_s = f"{st:,}".replace(",", " ") if st is not None else '-'
    print(f"  {str(date):<12} {com:<18} {dept:>4} {typ:<20} {sb_s:>8} {st_s:>8} {vf:>8,} {nat:>8}".replace(",", " "))

  Date         Commune            Dept Type                 Surf.bat Surf.ter     Prix   Nature
  ------------------------------------------------------------------------------------------
  2022-01-08   ILLIAT               01 (vide)                      -      103      100  Echange
  2022-01-08   ILLIAT               01 (vide)                      -       92      100  Echange
  2022-01-08   ILLIAT               01 (vide)                      -       19      100  Echange
  2022-01-27   MANTENAY-MONTLIN     01 (vide)                      -      155      100  Echange
  2022-01-27   MANTENAY-MONTLIN     01 (vide)                      -       13      100  Echange
  2022-01-04   FEILLENS             01 (vide)                      -      810      100    Vente
  2022-01-19   FEILLENS             01 (vide)                      -       41      100    Vente
  2022-02-02   CHALLES-LA-MONTAGNE   01 (vide)                      -    3 530      100    Vente
  2022-02-10   COLIGNY              01 (vi

## Cellule 8 — Biens batis a moins de 1 000 euros

Combien de maisons et d'appartements ont un prix inferieur a 1 000 euros ?
Ce sont les cas les plus susceptibles d'etre des erreurs ou des cessions atypiques.

In [8]:
r = con.execute(f"""
    SELECT
        \"Type local\",
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Surface reelle bati\"), 0) AS med_surface,
        ROUND(MEDIAN(\"Valeur fonciere\"), 0) AS med_prix
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < 1000
    AND \"Type local\" IN ('Maison', 'Appartement')
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchall()

print("Biens batis a moins de 1 000 euros :")
print(f"  {'Type':<20} {'Nb':>8} {'Med. surface':>14} {'Med. prix':>12}")
print(f"  {'-'*58}")
for typ, nb, ms, mp in r:
    print(f"  {typ:<20} {nb:>6,} {ms:>12,.0f} m2 {mp:>10,.0f} EUR".replace(",", " "))

print()
print("Exemples de maisons a moins de 1 000 euros :")
r2 = con.execute(f"""
    SELECT
        \"Date mutation\", \"Commune\", \"Code departement\",
        \"Surface reelle bati\", \"Valeur fonciere\", \"Nature mutation\"
    FROM '{pq}'
    WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < 1000
    AND \"Type local\" = 'Maison'
    ORDER BY \"Valeur fonciere\" ASC
    LIMIT 10
""").fetchall()
print(f"  {'Date':<12} {'Commune':<22} {'Dept':>4} {'Surface':>8} {'Prix':>8} {'Nature':>10}")
print(f"  {'-'*68}")
for date, com, dept, sb, vf, nat in r2:
    print(f"  {str(date):<12} {com:<22} {dept:>4} {sb:>6} m2 {vf:>6} EUR {nat:>10}")

Biens batis a moins de 1 000 euros :
  Type                       Nb   Med. surface    Med. prix
  ----------------------------------------------------------
  Appartement           2 780           59 m2          1 EUR
  Maison                1 250           82 m2          1 EUR

Exemples de maisons a moins de 1 000 euros :
  Date         Commune                Dept  Surface     Prix     Nature
  --------------------------------------------------------------------
  2022-02-17   SOURS                    28     49 m2      1 EUR      Vente
  2022-02-17   SOURS                    28     62 m2      1 EUR      Vente
  2022-02-17   SOURS                    28     62 m2      1 EUR      Vente
  2022-02-17   SOURS                    28     49 m2      1 EUR      Vente
  2022-02-17   SOURS                    28     62 m2      1 EUR      Vente
  2022-02-17   SOURS                    28     49 m2      1 EUR      Vente
  2022-02-17   SOURS                    28     62 m2      1 EUR      Vente
  2022

## Cellule 9 — Synthese de l'etape 2f

In [9]:
print("Synthese — Transactions a faible valeur DVF 2022")
print("=" * 55)
print()
for s in [10, 100, 1000, 5000]:
    r = con.execute(f"SELECT COUNT(*) FROM '{pq}' WHERE \"Valeur fonciere\" > 0 AND \"Valeur fonciere\" < {s}").fetchone()[0]
    pct = 100.0 * r / nb_avec_prix
    print(f"  < {s:>5,} EUR : {r:>8,} lignes ({pct:.2f} %)".replace(",", " "))

Synthese — Transactions a faible valeur DVF 2022

  <    10 EUR :   31 312 lignes (0.68 %)
  <   100 EUR :   37 212 lignes (0.81 %)
  < 1 000 EUR :   99 989 lignes (2.18 %)
  < 5 000 EUR :  265 529 lignes (5.79 %)


## Cellule 10 — Fermeture

In [10]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
